<a href="https://www.kaggle.com/code/cartelsmith/stacked-ensemble-model-mental-health-pred?scriptVersionId=327865353" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🎮 Gaming Addiction & Mental Health Risk: An Educational Walkthrough

Welcome! In this notebook, we're going to explore the relationship between gaming habits and mental health risk scores. Whether you're building out your data science portfolio or refining your machine learning pipelines, this dataset provides a fantastic sandbox. 

We will walk through:
1. **Exploratory Data Analysis (EDA)** to uncover trends and correlations.
2. **Data Cleaning & Feature Selection** to prep our data for modeling.
3. **Building an Ensemble Machine Learning Model** using Scikit-Learn to predict mental health risk.

Let's dive in and start coding!

## 1. Setup & Data Loading
First, let's load our necessary libraries and pull in the dataset. We'll be using `pandas` for data manipulation, `matplotlib` and `seaborn` for visualization, and `kagglehub` to directly download the dataset.

In [ ]:
# Install dependencies if running locally:
# !pip install kagglehub[pandas-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
from sklearn import set_config
set_config(display='diagram')

# Set visualization styles
sns.set_theme(style='white', context='notebook')

# Prevent wrapping dataframes across multiple lines for better readability
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

# Set the path to the file you'd like to load
file_path = "gaming_addiction.csv"

# Load the latest version of the dataset
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "ajitashwath/gaming-addiction-dataset",
  file_path,
)

## 2. Exploratory Data Analysis (EDA)
Before we build any models, we need to understand our data. Let's look at the first few rows and get a summary of the data types and potential missing values.

In [ ]:
# Let's peek at the first 5 rows
display(df.head())

# Check data types and non-null counts
print(df.info())

# Store column names for later reference
column_names = df.columns.tolist()

### Visualizing Feature Correlations
Machine learning models thrive on strong signals. Let's find out which features have the strongest correlation with our target variable: `mental_health_risk_score`. 

We'll visualize this using a horizontal bar plot (Pareto style) and a grid of scatter plots.

In [ ]:
# Calculate numerical correlations
df_correlations = df.corr(numeric_only=True)

# Grab the top 15 features most strongly correlated with mental health risk
# (We take 16 and skip the 1st one, which is the target correlated with itself)
mh_health_risk_corrs = df_correlations['mental_health_risk_score'].nlargest(16)

# Plot the top correlations
plt.figure(figsize=(10, 6))
sns.barplot(x=mh_health_risk_corrs[1:], y=mh_health_risk_corrs[1:].index, fill=True, orient='h', palette='Purples_r')
plt.title("Pareto of Correlations to Mental Health Risk Score")
plt.xlabel("Correlation Coefficient")
plt.show()

# Plotting scatter plots for the Top 15 parameters
fig, axes = plt.subplots(4, 4, figsize=(18, 15))
axes = axes.flatten()
mh_risk_cols = mh_health_risk_corrs.index.to_list()

for n, col in enumerate(mh_risk_cols[1:]): # Skip target variable itself
    sns.scatterplot(data=df, x=col, y='mental_health_risk_score', ax=axes[n], color='#73e8f7', alpha=0.6)
    axes[n].set_title(f'{col}')

plt.suptitle("Top Correlations to Mental Health Risk Score", fontsize=16)
plt.tight_layout()
plt.show()

## 3. Data Cleaning & Feature Selection
During EDA, we often find highly correlated features (multicollinearity). If two features provide the exact same information, keeping both just adds unnecessary noise to our model. 

For instance, `anxiety_level` and `depression_indicator` might be highly correlated. We can either feature-engineer a combined metric, or simply drop one. Here, we'll drop `depression_indicator`.

We'll also do a quick visual check to confirm we have no missing values.

In [ ]:
# Drop the redundant column
df_clean = df[mh_risk_cols].drop(columns=['depression_indicator'])

print(f'Here are the features we are moving forward with: \n{df_clean.columns.tolist()}\n')

# Visualize missing values using a heatmap
plt.figure(figsize=(10, 4))
sns.heatmap(df_clean.isna(), cmap='Dark2', cbar=False)
plt.ylabel('Row Number')
plt.xlabel('Column Name')
plt.title("Heatmap of Missing Values (Solid color means no missing data!)")
plt.show()

print("🙅🏽 Fantastic! There aren't any missing values in our feature set.")

## 4. Preparing for Machine Learning
Now, let's prepare to predict the `mental_health_risk_score`. 

We'll separate our data into **features (X)** and our **target (y)**. Then, we'll split the dataset into training and testing sets. Training data is used to teach our model, and testing data is held back to evaluate how well it learned.

Let's also visualize the distribution of our target variable.

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
features = df_clean.drop(columns=['mental_health_risk_score'])
target = df_clean['mental_health_risk_score']

print(f'Features shape: {features.shape}')
print(f'Target shape: {target.shape}')

# Visualize the target distribution
plt.figure(figsize=(8, 5))
sns.histplot(target, bins=20, color='#6555b4', kde=True)
plt.title("Target Variable Distribution: Mental Health Risk Score")
plt.xlabel("Risk Score")
plt.show()

# Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=12
)

## 5. Building a Stacking Regressor
Instead of relying on just one algorithm, we can combine several to create an **Ensemble Model**. 

We'll use a `StackingRegressor`. It trains several base models (like Decision Trees and Support Vector Machines) and then uses a "final estimator" (Linear Regression) to learn how to best combine their predictions. 

*Note: We also use `StandardScaler` in our pipeline to ensure all our features are on the same scale, which is crucial for algorithms like SVM and Ridge Regression.*

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import LinearSVR
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import StackingRegressor

# Define our base models
model_1 = DecisionTreeRegressor(random_state=42)
model_2 = LinearSVR(random_state=42)
model_3 = Ridge(random_state=42)
model_4 = LinearRegression()

# Combine them into a Stacking ensemble
ensemble_stack = StackingRegressor(
    estimators=[
        ('tree', model_1),
        ('svr', model_2),
        ('ridge', model_3),
        ('linreg', model_4)
    ],
    final_estimator=LinearRegression()
)

# Build a pipeline that scales data first, then feeds it to the stack
pipe = make_pipeline(StandardScaler(), ensemble_stack) 

# Train the model
pipe.fit(X_train, y_train)
print("✅ Model training complete!")
display(pipe)

## 6. Model Evaluation
Let's see how our model performs on the unseen test data. We'll use the $R^2$ score, which tells us the proportion of the variance in the target variable that is predictable from the features. An $R^2$ of 1.0 is perfect.

In [ ]:
# Evaluate on the test set
r2_score = pipe.score(X_test, y_test)

print(f'🎯 R^2 Score on Test Data: {r2_score:.3f}')

## 7. Conclusion & Future Work
This notebook provides a solid baseline for modeling mental health risk scores based on gaming habits! But there is always room to iterate and improve. 

**Challenge yourself to fork this notebook and try:**
* **Cross-Validation:** Since this is a smaller dataset, try using k-fold cross-validation to get a more robust performance metric.
* **Alternative Ensembles:** Build a `VotingRegressor` and compare its performance to this Stacked model.
* **Advanced Algorithms:** Implement bagging or boosting regressors (like `RandomForest` or `XGBoost`) to see if they can beat our score.

*Click 'Copy & Edit' to get started. Happy Coding!*